# 01 — Dataset Exploratory Data Analysis

**Purpose:** Understand the structure, distributions, and temporal patterns of the engineered ENSO dataset before training any models.

This notebook covers:
1. Dataset overview and basic statistics
2. ENSO phase distribution and temporal balance
3. Feature correlation structure
4. Time series visualisation of key indices
5. Cross-correlation between features and future targets (lag analysis)
6. Missing value audit


In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
PROCESSED = Path('../../data/processed')

In [ ]:
df = pd.read_parquet(PROCESSED / 'enso_dataset.parquet').set_index('date')
print(f'Shape: {df.shape}')
print(f'Date range: {df.index[0].date()} → {df.index[-1].date()}')
df.head()

## 1. ENSO phase distribution

In [ ]:
from src.evaluation.plots import plot_class_distribution
fig = plot_class_distribution(df, col='enso_phase')
plt.show()

print('\nClass balance:')
print(df['enso_phase'].value_counts(normalize=True).mul(100).round(1).to_string())

## 2. Niño 3.4 time series with phase overlay

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.axhline(0.5,  color='#d6604d', linestyle='--', linewidth=0.8, label='El Niño threshold')
ax.axhline(-0.5, color='#4393c3', linestyle='--', linewidth=0.8, label='La Niña threshold')
ax.fill_between(df.index, df['nino34_anom'], 0,
                where=(df['nino34_anom'] > 0.5),  color='#d6604d', alpha=0.3)
ax.fill_between(df.index, df['nino34_anom'], 0,
                where=(df['nino34_anom'] < -0.5), color='#4393c3', alpha=0.3)
ax.plot(df.index, df['nino34_anom'], color='black', linewidth=0.8)
ax.set_ylabel('SST anomaly (°C)')
ax.set_title('Niño 3.4 anomaly (1980–present)', fontweight='bold')
ax.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 3. Feature correlation matrix

In [ ]:
from src.feature_engineering.builder import get_feature_columns
feat_cols = get_feature_columns(df)
numeric_feats = df[feat_cols].select_dtypes(include='number')

corr = numeric_feats.corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=True, yticklabels=True, ax=ax,
            linewidths=0.1, square=True)
ax.set_title('Feature correlation matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Cross-correlation: features vs future target

In [ ]:
label_enc = {'El Niño': 2, 'Neutral': 1, 'La Niña': 0}
for target in ['enso_t1', 'enso_t3', 'enso_t6']:
    y = df[target].map(label_enc)
    corrs = numeric_feats.corrwith(y).abs().sort_values(ascending=False).head(10)
    print(f'\nTop 10 features correlated with {target}:')
    print(corrs.to_string())

## 5. Missing value audit

In [ ]:
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
if len(missing):
    print('Columns with missing values:')
    print(missing.to_string())
else:
    print('No missing values in dataset ✓')